<a href="https://colab.research.google.com/github/matiashhidalgo/metodos_multivariados/blob/main/ANOVA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

datos = pd.DataFrame({
    'Estudiante': range(1, 11),
    'Horas': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Nota': [3.2, 4.5, 5.0, 6.8, 7.0, 8.5, 9.0, 9.2, 9.8, 10.0]
})

n = len(datos)
x = datos['Horas'].values
y = datos['Nota'].values
datos

,Estudiante,Horas,Nota
0,1,1,3.2
1,2,2,4.5
2,3,3,5.0
3,4,4,6.8
4,5,5,7.0
5,6,6,8.5
6,7,7,9.0
7,8,8,9.2
8,9,9,9.8
9,10,10,10.0


In [ ]:
x_bar = x.mean()
y_bar = y.mean()

Sxy = np.sum((x - x_bar) * (y - y_bar))
Sxx = np.sum((x - x_bar) ** 2)

b1 = Sxy / Sxx
b0 = y_bar - b1 * x_bar

y_hat = b0 + b1 * x

print(f"b0 = {b0:.4f}")
print(f"b1 = {b1:.4f}")

b0 = 3.0533
b1 = 0.7721


In [ ]:
SST = np.sum((y - y_bar) ** 2)
SSR = np.sum((y_hat - y_bar) ** 2)
SSE = np.sum((y - y_hat) ** 2)

df_reg = 1
df_err = n - 2
df_tot = n - 1

MSR = SSR / df_reg
MSE = SSE / df_err

F_stat = MSR / MSE

anova = pd.DataFrame({
    'Fuente': ['Regresión', 'Error', 'Total'],
    'SC': [SSR, SSE, SST],
    'gl': [df_reg, df_err, df_tot],
    'CM': [MSR, MSE, np.nan],
    'F': [F_stat, np.nan, np.nan]
})
anova

,Fuente,SC,gl,CM,F
0,Regresión,49.184121,1,49.184121,152.752906
1,Error,2.575879,8,0.321985,NaN
2,Total,51.760000,9,NaN,NaN


In [ ]:
alpha = 0.05

# Prueba F
F_crit = stats.f.ppf(1 - alpha, df_reg, df_err)
p_valor_F = 1 - stats.f.cdf(F_stat, df_reg, df_err)

# Prueba t para b1
SE_b1 = np.sqrt(MSE / Sxx)
t_b1 = b1 / SE_b1
t_crit = stats.t.ppf(1 - alpha/2, df_err)
p_valor_t = 2 * (1 - stats.t.cdf(abs(t_b1), df_err))

# Prueba t para b0
SE_b0 = np.sqrt(MSE * (1/n + x_bar**2 / Sxx))
t_b0 = b0 / SE_b0
p_valor_t0 = 2 * (1 - stats.t.cdf(abs(t_b0), df_err))

print(f"F = {F_stat:.4f}, F_crit = {F_crit:.4f}, p = {p_valor_F:.6f}")
print(f"t(b1) = {t_b1:.4f}, t_crit = {t_crit:.4f}, p = {p_valor_t:.6f}, SE(b1) = {SE_b1:.4f}")
print(f"t(b0) = {t_b0:.4f}, p = {p_valor_t0:.6f}, SE(b0) = {SE_b0:.4f}")

F = 152.7529, F_crit = 5.3177, p = 0.000002
t(b1) = 12.3593, t_crit = 2.3060, p = 0.000002, SE(b1) = 0.0625
t(b0) = 7.8769, p = 0.000049, SE(b0) = 0.3876


In [ ]:
print(f"t(b1)^2 = {t_b1**2:.4f}")
print(f"F        = {F_stat:.4f}")
print(f"¿Coinciden? {np.isclose(t_b1**2, F_stat)}")

t(b1)^2 = 152.7529
F        = 152.7529
¿Coinciden? True


In [ ]:
IC_b1 = (b1 - t_crit * SE_b1, b1 + t_crit * SE_b1)
IC_b0 = (b0 - t_crit * SE_b0, b0 + t_crit * SE_b0)

print(f"IC 95% b1: ({IC_b1[0]:.4f}, {IC_b1[1]:.4f})")
print(f"IC 95% b0: ({IC_b0[0]:.4f}, {IC_b0[1]:.4f})")

IC 95% b1: (0.6281, 0.9162)
IC 95% b0: (2.1594, 3.9472)


In [ ]:
SE_media = np.sqrt(MSE * (1/n + (x - x_bar)**2 / Sxx))
SE_pred  = np.sqrt(MSE * (1 + 1/n + (x - x_bar)**2 / Sxx))

tabla_ic = pd.DataFrame({
    'Horas': x,
    'Nota': y,
    'y_hat': y_hat,
    'IC_inf': y_hat - t_crit * SE_media,
    'IC_sup': y_hat + t_crit * SE_media,
    'IP_inf': y_hat - t_crit * SE_pred,
    'IP_sup': y_hat + t_crit * SE_pred
})
tabla_ic

,Horas,Nota,y_hat,IC_inf,IC_sup,IP_inf,IP_sup
0,1,3.2,3.825455,3.056371,4.594538,2.307662,5.343247
1,2,4.5,4.597576,3.945305,5.249846,3.135501,6.059650
2,3,5.0,5.369697,4.821123,5.918270,3.950846,6.788548
3,4,6.8,6.141818,5.675002,6.608634,4.752530,7.531106
4,5,7.0,6.913939,6.493929,7.333950,5.539671,8.288208
5,6,8.5,7.686061,7.266050,8.106071,6.311792,9.060329
6,7,9.0,8.458182,7.991366,8.924998,7.068894,9.847470
7,8,9.2,9.230303,8.681730,9.778877,7.811452,10.649154
8,9,9.8,10.002424,9.350154,10.654695,8.540350,11.464499
9,10,10.0,10.774545,10.005462,11.543629,9.256753,12.292338
